# Sink vuln final decisions parser

Reads only final `sink_vuln_*` result folders from `outputs/sweep`, extracts `summary`, `vulnerability_probability`, `follow_up_checks`, and writes CSV tables.

In [2]:
import csv
import json
from pathlib import Path

ROOT = Path('/Users/nident/Desktop/JOB/ScolTech/CPG_analysis')
OUTPUTS_DIR = ROOT / 'outputs' / 'sweep'
REPORT_DIR = ROOT / 'outputs' / 'sweep_reports'
THRESHOLD = 0.4

REPORT_DIR.mkdir(parents=True, exist_ok=True)

decision_rows = []
commit_rows = []

sink_vuln_dirs = sorted(OUTPUTS_DIR.glob('*/sink_vuln_*'))

for sink_vuln_dir in sink_vuln_dirs:
    if not sink_vuln_dir.is_dir():
        continue

    project_key = sink_vuln_dir.parent.name
    dataset_key = sink_vuln_dir.name.removeprefix('sink_vuln_')
    if '__' not in dataset_key:
        continue

    dataset_project_key, commit = dataset_key.rsplit('__', 1)
    if dataset_project_key != project_key:
        project_key = dataset_project_key

    if '__' in project_key:
        project, repository = project_key.split('__', 1)
    else:
        project, repository = project_key, ''

    ok_files = sorted(sink_vuln_dir.glob('*_ok.json'))
    error_files = sorted(sink_vuln_dir.glob('*_error.json'))

    probabilities = []
    vulnerable_decisions = []
    all_summaries = []
    all_follow_up_checks = []

    for path in ok_files:
        with path.open('r', encoding='utf-8') as f:
            data = json.load(f)

        analysis = data.get('analysis') or {}
        probability = analysis.get('vulnerability_probability')
        if isinstance(probability, (int, float)):
            probability = float(probability)
            probabilities.append(probability)
        else:
            probability = None

        summary = analysis.get('summary') or ''
        follow_up_checks = analysis.get('follow_up_checks') or []
        source_node_id = data.get('sourceNodeId') or data.get('nodeId')
        path_assessments = analysis.get('path_assessments') or []

        if summary:
            all_summaries.append(summary)
        for check in follow_up_checks:
            all_follow_up_checks.append(check)

        max_path_probability = None
        path_reasonings = []
        sink_node_ids = []
        sink_kinds = []

        for assessment in path_assessments:
            path_probability = assessment.get('vulnerability_probability')
            if isinstance(path_probability, (int, float)):
                path_probability = float(path_probability)
                max_path_probability = path_probability if max_path_probability is None else max(max_path_probability, path_probability)
            if assessment.get('reasoning'):
                path_reasonings.append({
                    'path_index': assessment.get('path_index'),
                    'sink_node_id': assessment.get('sink_node_id'),
                    'sink_kind': assessment.get('sink_kind'),
                    'vulnerability_probability': path_probability,
                    'reasoning': assessment.get('reasoning'),
                })
            if assessment.get('sink_node_id') is not None:
                sink_node_ids.append(assessment.get('sink_node_id'))
            if assessment.get('sink_kind'):
                sink_kinds.append(assessment.get('sink_kind'))

        is_potentially_vulnerable = probability is not None and probability > THRESHOLD
        if is_potentially_vulnerable:
            vulnerable_decisions.append({
                'source_node_id': source_node_id,
                'vulnerability_probability': probability,
                'summary': summary,
                'follow_up_checks': follow_up_checks,
                'file': str(path.relative_to(ROOT)),
            })

        decision_rows.append({
            'project': project,
            'repository': repository,
            'project_key': project_key,
            'commit': commit,
            'dataset_key': dataset_key,
            'source_node_id': source_node_id,
            'vulnerability_probability': probability,
            'is_potentially_vulnerable': is_potentially_vulnerable,
            'summary': summary,
            'follow_up_checks': json.dumps(follow_up_checks, ensure_ascii=False),
            'max_path_probability': max_path_probability,
            'sink_node_ids': json.dumps(sorted(set(sink_node_ids)), ensure_ascii=False),
            'sink_kinds': json.dumps(sorted(set(sink_kinds)), ensure_ascii=False),
            'path_reasonings': json.dumps(path_reasonings, ensure_ascii=False),
            'sink_vuln_file': str(path.relative_to(ROOT)),
        })

    first_error_messages = []
    for path in error_files[:5]:
        with path.open('r', encoding='utf-8') as f:
            data = json.load(f)
        first_error_messages.append((data.get('error') or {}).get('message'))

    commit_rows.append({
        'project': project,
        'repository': repository,
        'project_key': project_key,
        'commit': commit,
        'dataset_key': dataset_key,
        'threshold': THRESHOLD,
        'sink_vuln_ok_count': len(ok_files),
        'sink_vuln_error_count': len(error_files),
        'max_vulnerability_probability': max(probabilities) if probabilities else None,
        'avg_vulnerability_probability': sum(probabilities) / len(probabilities) if probabilities else None,
        'potentially_vulnerable_count': len(vulnerable_decisions),
        'summaries': json.dumps(all_summaries, ensure_ascii=False),
        'follow_up_checks': json.dumps(all_follow_up_checks, ensure_ascii=False),
        'potentially_vulnerable_decisions': json.dumps(vulnerable_decisions, ensure_ascii=False),
        'first_error_messages': json.dumps([x for x in first_error_messages if x], ensure_ascii=False),
        'sink_vuln_dir': str(sink_vuln_dir.relative_to(ROOT)),
    })

commit_csv = REPORT_DIR / 'sink_vuln_commit_summary.csv'
decision_csv = REPORT_DIR / 'sink_vuln_decisions.csv'

commit_columns = [
    'project', 'repository', 'project_key', 'commit', 'dataset_key', 'threshold',
    'sink_vuln_ok_count', 'sink_vuln_error_count',
    'max_vulnerability_probability', 'avg_vulnerability_probability', 'potentially_vulnerable_count',
    'summaries', 'follow_up_checks', 'potentially_vulnerable_decisions', 'first_error_messages', 'sink_vuln_dir',
]

decision_columns = [
    'project', 'repository', 'project_key', 'commit', 'dataset_key', 'source_node_id',
    'vulnerability_probability', 'is_potentially_vulnerable', 'summary', 'follow_up_checks',
    'max_path_probability', 'sink_node_ids', 'sink_kinds', 'path_reasonings', 'sink_vuln_file',
]

with commit_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=commit_columns)
    writer.writeheader()
    writer.writerows(commit_rows)

with decision_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=decision_columns)
    writer.writeheader()
    writer.writerows(decision_rows)

print(f'sink_vuln dirs parsed: {len(sink_vuln_dirs)}')
print(f'commit summary rows: {len(commit_rows)} -> {commit_csv}')
print(f'final decision rows: {len(decision_rows)} -> {decision_csv}')
print(f'potentially vulnerable decisions > {THRESHOLD}: {sum(1 for r in decision_rows if r["is_potentially_vulnerable"])}')
print('commits per project:')
for project_key in sorted(set(row['project_key'] for row in commit_rows)):
    commits = sorted(row['commit'] for row in commit_rows if row['project_key'] == project_key)
    print(f'  {project_key}: {len(commits)} -> {commits}')

try:
    import pandas as pd
    display(pd.DataFrame(commit_rows).sort_values(['potentially_vulnerable_count', 'max_vulnerability_probability'], ascending=[False, False]).head(50))
    display(pd.DataFrame(decision_rows).sort_values(['vulnerability_probability'], ascending=False).head(50))
except Exception:
    print('pandas is not installed; CSV files were written successfully.')
    for row in sorted(commit_rows, key=lambda r: (r['potentially_vulnerable_count'], r['max_vulnerability_probability'] or -1), reverse=True)[:20]:
        print(row['project_key'], row['commit'], row['potentially_vulnerable_count'], row['max_vulnerability_probability'])


sink_vuln dirs parsed: 22
commit summary rows: 22 -> /Users/nident/Desktop/JOB/ScolTech/CPG_analysis/outputs/sweep_reports/sink_vuln_commit_summary.csv
final decision rows: 190 -> /Users/nident/Desktop/JOB/ScolTech/CPG_analysis/outputs/sweep_reports/sink_vuln_decisions.csv
potentially vulnerable decisions > 0.4: 0
commits per project:
  CycloneDX__cyclonedx-core-java: 2 -> ['7c7294b3d112', 'af0ec75c93c0']
  FasterXML__jackson-core: 4 -> ['4e445083ddde', '8b25fd67f205', 'a6c297682737', 'ad93650d6f13']
  allure-framework__allure2: 2 -> ['cbcb33719851', 'eaa87ff7d93e']
  apache__jspwiki: 1 -> ['a184e0a53d1c']
  gaul__s3proxy: 2 -> ['83b159ee4b9f', '86b6ee4749aa']
  gematik__app-referencevalidator: 2 -> ['d276b02be9c8', 'd6d27613fab7']
  jenkinsci__delphix-plugin: 2 -> ['646851528039', '798a36148526']
  jenkinsci__gatling-plugin: 2 -> ['141bd3a811ab', '83a08aaa7d33']
  jenkinsci__htmlpublisher-plugin: 2 -> ['6b840248dd0d', 'c0eed940e65e']
  jenkinsci__structs-plugin: 2 -> ['1b04ea4df7c8', 

,project,repository,project_key,commit,dataset_key,threshold,sink_vuln_ok_count,sink_vuln_error_count,max_vulnerability_probability,avg_vulnerability_probability,potentially_vulnerable_count,summaries,follow_up_checks,potentially_vulnerable_decisions,first_error_messages,sink_vuln_dir
7,allure-framework,allure2,allure-framework__allure2,eaa87ff7d93e,allure-framework__allure2__eaa87ff7d93e,0.4,21,0,0.1,0.004762,0,"[""The source is command-line arguments to a Ja...","[""Trace dataflow from args through the JComman...",[],[],outputs/sweep/allure-framework__allure2/sink_v...
8,apache,jspwiki,apache__jspwiki,a184e0a53d1c,apache__jspwiki__a184e0a53d1c,0.4,82,96,0.1,0.007317,0,"[""The source is an HTTP request parameter (ANT...","[""Verify that the sink rule 'python-filesystem...",[],"[""Error code: 402 - {'error': {'message': 'Ins...",outputs/sweep/apache__jspwiki/sink_vuln_apache...
21,x-stream,xstream,x-stream__xstream,15cb057ecef4,x-stream__xstream__15cb057ecef4,0.4,28,29,0.1,0.003571,0,"[""The source is a command-line argument parame...","[""Verify that the benchmark test class is not ...",[],"[""Error code: 402 - {'error': {'message': 'Ins...",outputs/sweep/x-stream__xstream/sink_vuln_x-st...
2,FasterXML,jackson-core,FasterXML__jackson-core,4e445083ddde,FasterXML__jackson-core__4e445083ddde,0.4,10,0,0.0,0.000000,0,"[""The source is a JVM system property read use...","[""Verify that the system property 'SYSTEM_PROP...",[],[],outputs/sweep/FasterXML__jackson-core/sink_vul...
3,FasterXML,jackson-core,FasterXML__jackson-core,8b25fd67f205,FasterXML__jackson-core__8b25fd67f205,0.4,9,0,0.0,0.000000,0,"[""The source is a JVM system property read (li...","[""Verify that the sink rule 'python-write-call...",[],[],outputs/sweep/FasterXML__jackson-core/sink_vul...
4,FasterXML,jackson-core,FasterXML__jackson-core,a6c297682737,FasterXML__jackson-core__a6c297682737,0.4,10,0,0.0,0.000000,0,"[""The source is a JVM system property read in ...","[""Verify that the system property SYSTEM_PROPE...",[],[],outputs/sweep/FasterXML__jackson-core/sink_vul...
5,FasterXML,jackson-core,FasterXML__jackson-core,ad93650d6f13,FasterXML__jackson-core__ad93650d6f13,0.4,9,0,0.0,0.000000,0,"[""The source is System.getProperty(\""line.sepa...","[""Verify that the 'eol' and 'indents' fields i...",[],[],outputs/sweep/FasterXML__jackson-core/sink_vul...
6,allure-framework,allure2,allure-framework__allure2,cbcb33719851,allure-framework__allure2__cbcb33719851,0.4,21,0,0.0,0.000000,0,"[""The source is the command-line arguments of ...","[""Verify that the JCommander parsing in comman...",[],[],outputs/sweep/allure-framework__allure2/sink_v...
0,CycloneDX,cyclonedx-core-java,CycloneDX__cyclonedx-core-java,7c7294b3d112,CycloneDX__cyclonedx-core-java__7c7294b3d112,0.4,0,0,NaN,NaN,0,[],[],[],[],outputs/sweep/CycloneDX__cyclonedx-core-java/s...
1,CycloneDX,cyclonedx-core-java,CycloneDX__cyclonedx-core-java,af0ec75c93c0,CycloneDX__cyclonedx-core-java__af0ec75c93c0,0.4,0,0,NaN,NaN,0,[],[],[],[],outputs/sweep/CycloneDX__cyclonedx-core-java/s...


,project,repository,project_key,commit,dataset_key,source_node_id,vulnerability_probability,is_potentially_vulnerable,summary,follow_up_checks,max_path_probability,sink_node_ids,sink_kinds,path_reasonings,sink_vuln_file
142,apache,jspwiki,apache__jspwiki,a184e0a53d1c,apache__jspwiki__a184e0a53d1c,248035,0.1,False,The source is an HTTP request URI obtained via...,"[""Verify that ThreadContext.remove() does not ...",0.0,"[104431, 232187, 242576, 245628, 254011, 27004...","[""filesystem_mutation"", ""sql_execution""]","[{""path_index"": 0, ""sink_node_id"": 242576, ""si...",outputs/sweep/apache__jspwiki/sink_vuln_apache...
148,apache,jspwiki,apache__jspwiki,a184e0a53d1c,apache__jspwiki__a184e0a53d1c,242352,0.1,False,The source is an HTTP request URI obtained via...,"[""Verify whether the request URI is used in an...",0.1,"[104456, 242495, 242576, 245628, 252055, 27004...","[""filesystem_mutation"", ""filesystem_write"", ""s...","[{""path_index"": 0, ""sink_node_id"": 242576, ""si...",outputs/sweep/apache__jspwiki/sink_vuln_apache...
175,x-stream,xstream,x-stream__xstream,15cb057ecef4,x-stream__xstream__15cb057ecef4,179952,0.1,False,The source is a JVM system property 'xstream.d...,"[""Verify if the Class.forName(driver) call in ...",0.0,"[2095, 6313, 8730, 8734, 77140, 77181, 148313,...","[""filesystem_mutation"", ""filesystem_write""]","[{""path_index"": 0, ""sink_node_id"": 8734, ""sink...",outputs/sweep/x-stream__xstream/sink_vuln_x-st...
85,apache,jspwiki,apache__jspwiki,a184e0a53d1c,apache__jspwiki__a184e0a53d1c,42541,0.1,False,The source is a JVM system property read in a ...,"[""Search for uses of TESTS_CONFIG_DOWNLOADS_FO...",NaN,[],[],[],outputs/sweep/apache__jspwiki/sink_vuln_apache...
105,apache,jspwiki,apache__jspwiki,a184e0a53d1c,apache__jspwiki__a184e0a53d1c,423436,0.1,False,The source is an HTTP request URI from a servl...,"[""Inspect the XML-RPC server implementation (o...",0.2,"[104431, 232187, 242576, 245628, 254011, 27004...","[""filesystem_mutation"", ""sql_execution""]","[{""path_index"": 0, ""sink_node_id"": 242576, ""si...",outputs/sweep/apache__jspwiki/sink_vuln_apache...
149,apache,jspwiki,apache__jspwiki,a184e0a53d1c,apache__jspwiki__a184e0a53d1c,242576,0.1,False,The source is a ThreadContext.remove call in a...,"[""Inspect how ThreadContext keys are used else...",0.1,[242576],"[""filesystem_mutation""]","[{""path_index"": 0, ""sink_node_id"": 242576, ""si...",outputs/sweep/apache__jspwiki/sink_vuln_apache...
72,allure-framework,allure2,allure-framework__allure2,eaa87ff7d93e,allure-framework__allure2__eaa87ff7d93e,100566,0.1,False,The source is a servlet request input stored i...,"[""Manually inspect how this.parameters is used...",NaN,[],[],[],outputs/sweep/allure-framework__allure2/sink_v...
134,apache,jspwiki,apache__jspwiki,a184e0a53d1c,apache__jspwiki__a184e0a53d1c,344333,0.1,False,The source is an HTTP request InputStream in A...,"[""Examine the Sandler.unmarshallEntry implemen...",0.1,"[104431, 232187, 245628, 254011, 270042, 27980...","[""filesystem_mutation"", ""sql_execution""]","[{""path_index"": 0, ""sink_node_id"": 270042, ""si...",outputs/sweep/apache__jspwiki/sink_vuln_apache...
128,apache,jspwiki,apache__jspwiki,a184e0a53d1c,apache__jspwiki__a184e0a53d1c,229947,0.0,False,"The source is System.getProperty(""user.home"") ...","[""Verify if m_storageDir (assigned from the so...",0.0,"[8293, 15029, 30758, 85036, 85270, 210259, 232...","[""filesystem_mutation"", ""template_rendering""]","[{""path_index"": 0, ""sink_node_id"": 232187, ""si...",outputs/sweep/apache__jspwiki/sink_vuln_apache...
121,apache,jspwiki,apache__jspwiki,a184e0a53d1c,apache__jspwiki__a184e0a53d1c,351440,0.0,False,The source is the HTTP Referer header read in ...,"[""Verify the actual use of the 'referrer' vari...",0.0,"[104431, 232187, 245628, 254011, 270042, 27980...","[""filesystem_mutation"", ""sql_execution""]","[{""path_index"": 0, ""sink_node_id"": 388803, ""si...",outputs/sweep/apache__jspwiki/sink_vuln_apache...
